<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L03-evaluating-agents/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L03-evaluating-agents/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L03-evaluating-agents/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L03-evaluating-agents/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T07-L03 · Evaluating agents: reliability you can report

**You will build:** independently seeded repeated trials per task; the pass@k and pass^k
reliability estimators, sourced from their own papers; trajectory metrics and a failure
taxonomy computed straight from traces; and a paired bootstrap that resamples TASKS, not
trials, to compare two agent configurations with an interval on the difference.

**Time:** ~50 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track, T07-L01-agent-loop-from-scratch.

"It worked when I tried it" is not an evaluation. Section 1 carries T07-L01's harness,
stand-in policy, tool registry and trace format forward unchanged — this lesson is about
what a team does with MANY runs of it, not about rebuilding it.

By the end you will be able to:

1. Implement seeded, reproducible repeated trials per task, and aggregate them into per-task
   success counts.
2. Implement the pass@k and pass^k reliability estimators from their primary sources, and
   measure that pass@k rises with k while pass^k falls on the same trial data.
3. Compute trajectory metrics — steps, tool errors, recoveries and cost per success — and a
   failure taxonomy, directly from agent traces.
4. Measure how much narrower — and wrong — a bootstrap over pooled trials makes a suite's
   95% interval look than one over tasks, and explain why the task is the unit to resample.
5. Implement a paired bootstrap comparison of two agent configurations on the same tasks and
   seed, and decide whether the paired difference in reliability is distinguishable from
   zero.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `T07-L01`

**This lesson assumes you can already:**

- compute a mean and a percentile of a list with numpy, and draw samples with replacement from a seeded Generator
- work out the probability that k independent trials in a row all succeed at success rate p, and read a binomial coefficient C(n, k)
- loop over a list of trace records (dicts) and count outcomes with a dict or Counter

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Evaluating Large Language Models Trained on Code](https://arxiv.org/abs/2107.03374) · Chen et al. (arXiv) · *free* — section 2.1, Functional Correctness, defines pass@k and its unbiased estimator, which exercise 3 implements
- [tau-bench: A Benchmark for Tool-Agent-User Interaction in Real-World Domains](https://arxiv.org/abs/2406.12045) · Yao, Shinn, Razavi and Narasimhan (arXiv) · *free* — where pass^k, the probability that all k trials succeed, is proposed as an agent reliability metric (exercise 4)
- [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) · Anthropic · *free* — trials, graders, and why pass@k and pass^k answer different questions when agent runs vary
- [Think Stats, 2nd edition](https://greenteapress.com/thinkstats2/html/thinkstats2009.html) · Allen B. Downey (O'Reilly; free from Green Tea Press) · *free* — chapter 8, Estimation: sampling distributions, standard errors and confidence intervals found by simulation, the idea you apply when bootstrapping over tasks
- [An Introduction to the Bootstrap](https://www.routledge.com/An-Introduction-to-the-Bootstrap/Efron-Tibshirani/p/book/9780412042317) · Bradley Efron and R. J. Tibshirani (Chapman & Hall/CRC) · *paid* — the chapters on bootstrap standard errors and percentile confidence intervals, behind sections 5 and 11

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import random
import sys
import time
import traceback
from typing import Any, Callable, NamedTuple, Sequence

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20260924          # this lesson's one fixed seed: every task, every trial, every fault
                          # and every bootstrap resample derives from it, so re-running prints
                          # the same numbers on any machine.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("run_suite_trials",),
    "exercise 2": ("task_success_counts",),
    "exercise 3": ("pass_at_k",),
    "exercise 4": ("pass_hat_k",),
    "exercise 5": ("trajectory_metrics",),
    "exercise 6": ("failure_taxonomy_from_traces",),
    "exercise 7": ("paired_diff_interval",),
    "exercise 8": ("compare_configurations",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (pass_at_k)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 10     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 2048     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. Carried forward from T07-L01, unchanged

`ToolRegistry`, `validate_args`, `call_tool`, `backoff_delay`, `call_tool_with_retries`,
`is_stuck`, `budget_status`, `ScriptedPolicy` and `run_agent_loop` are T07-L01's code, copied
unchanged — the functions you built there and the ones it gave you — as working code: this
lesson is not about rebuilding them, it is about what a team does with many runs of them. If a name below is unfamiliar, T07-L01
is where it was built and explained; nothing here calls a language model either.

In [ ]:
class TransientToolError(Exception):
    """Raised by a tool for a failure worth retrying (a timeout, a dropped connection).

    Never raised for a mistake the CALLER made — a missing argument is never transient.
    """


class ToolResult(NamedTuple):
    ok: bool
    output: Any
    error: str | None
    transient: bool     # only meaningful when ok is False: worth a retry, or not?


class ToolRegistry(NamedTuple):
    tools: dict[str, Callable[..., Any]]
    schemas: dict[str, dict]


TOOL_SCHEMAS: dict[str, dict] = {
    "lookup": {"type": "object", "properties": {"key": {"type": "string"}},
              "required": ["key"]},
    "submit": {"type": "object", "properties": {"value": {"type": "integer"}},
              "required": ["value"]},
}


class ChainTask(NamedTuple):
    task_id: int
    start_key: str
    key_sequence: list[str]          # the correct keys, in order; key_sequence[0] == start_key
    chain: dict[str, Any]            # key -> next key (str), or the terminal value (int)
    target: int
    length: int                      # == len(key_sequence)


def generate_chain_task(length: int, seed: int) -> ChainTask:
    """One deterministic lookup chain of the given length."""
    rng = random.Random(seed)
    keys = [f"k{i}_{rng.randrange(1_000_000):06d}" for i in range(length)]
    chain: dict[str, Any] = {keys[i]: keys[i + 1] for i in range(length - 1)}
    target = rng.randint(100, 999)
    chain[keys[-1]] = target
    return ChainTask(task_id=seed, start_key=keys[0], key_sequence=keys, chain=chain,
                     target=target, length=length)


def generate_task_suite(lengths: Sequence[int], n_per_length: int, seed: int) -> list[ChainTask]:
    """`n_per_length` tasks at each length in `lengths`, all deterministic given `seed`."""
    tasks, task_id = [], 0
    for length in lengths:
        for _ in range(n_per_length):
            tasks.append(generate_chain_task(length, seed=seed * 1000 + task_id))
            task_id += 1
    return tasks


def make_registry(chain: dict, target: int, transient_rate: float,
                  rng: random.Random) -> ToolRegistry:
    """A registry over ONE task's chain. `lookup` fails transiently at `transient_rate`."""
    def lookup(key):
        if rng.random() < transient_rate:
            raise TransientToolError("lookup timed out")
        if key not in chain:
            raise ValueError(f"unknown key {key!r}")
        val = chain[key]
        return {"next_key": val} if isinstance(val, str) else {"value": val}

    def submit(value):
        return {"correct": value == target}

    return ToolRegistry(tools={"lookup": lookup, "submit": submit}, schemas=TOOL_SCHEMAS)


_ARG_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool,
             "array": list, "object": dict}


def validate_args(schema: dict, args: dict) -> list[str]:
    """Check `args` against a documented subset of JSON Schema, by hand. Never raises."""
    errors: list[str] = []
    for name in schema.get("required", []):
        if name not in args:
            errors.append(f"missing required argument {name!r}")
    properties = schema.get("properties", {})
    for name, value in args.items():
        prop = properties.get(name)
        if prop is None:
            continue
        expected = prop.get("type")
        py_type = _ARG_TYPES.get(expected)
        if py_type is not None:
            reject = (isinstance(value, bool) if expected in ("integer", "number") else False)
            if reject or not isinstance(value, py_type):
                errors.append(f"argument {name!r} should be of type {expected!r}, got {value!r}")
                continue
        if "enum" in prop and value not in prop["enum"]:
            errors.append(f"argument {name!r} must be one of {prop['enum']!r}, got {value!r}")
    return errors


def call_tool(registry: ToolRegistry, name: str, args: dict, validate: bool = True) -> ToolResult:
    """Execute one tool call through `registry`, never letting an exception escape."""
    if name not in registry.tools:
        return ToolResult(ok=False, output=None, error=f"unknown tool {name!r}", transient=False)
    schema = registry.schemas.get(name, {})
    if validate:
        errors = validate_args(schema, args)
        if errors:
            return ToolResult(ok=False, output=None,
                              error="invalid arguments: " + "; ".join(errors), transient=False)
        try:
            return ToolResult(ok=True, output=registry.tools[name](**args), error=None,
                              transient=False)
        except TransientToolError as exc:
            return ToolResult(ok=False, output=None, error=str(exc), transient=True)
        except Exception as exc:
            return ToolResult(ok=False, output=None,
                              error=f"tool {name!r} raised {type(exc).__name__}: {exc}",
                              transient=False)
    try:
        return ToolResult(ok=True, output=registry.tools[name](**args), error=None,
                          transient=False)
    except Exception as exc:
        return ToolResult(ok=False, output=None,
                          error=f"tool {name!r} raised {type(exc).__name__}: {exc}",
                          transient=True)


BASE_DELAY = 0.05    # seconds (simulated — nothing in this lesson actually sleeps)
CAP_DELAY = 2.0


def backoff_delay(attempt: int, base: float, cap: float, rng: random.Random) -> float:
    """Full-jitter capped exponential backoff. `attempt` is 0 for the FIRST retry."""
    ceiling = min(cap, base * (2 ** attempt))
    return rng.uniform(0.0, ceiling)


def call_tool_with_retries(registry: ToolRegistry, name: str, args: dict, max_retries: int,
                           rng: random.Random, validate: bool = True) -> tuple[ToolResult, int, float]:
    """Call a tool, retrying only a TRANSIENT failure, up to `max_retries` times."""
    attempt = 0
    total_backoff = 0.0
    result = call_tool(registry, name, args, validate=validate)
    while not result.ok and result.transient and attempt < max_retries:
        total_backoff += backoff_delay(attempt, BASE_DELAY, CAP_DELAY, rng)
        attempt += 1
        result = call_tool(registry, name, args, validate=validate)
    return result, attempt + 1, total_backoff


def is_stuck(trace: list[dict], window: int) -> bool:
    """True if the LAST `window` steps of `trace` are all the same action with the same outcome."""
    if len(trace) < window:
        return False
    last = trace[-window:]

    def signature(entry):
        return entry["tool"], tuple(sorted(entry["args"].items())), entry["ok"]

    first = signature(last[0])
    return all(signature(entry) == first for entry in last)


def budget_status(step: int, cost_spent: int, max_steps: int, max_cost: int) -> str | None:
    """Which budget, if any, is exhausted — checked in this fixed order: step, then cost."""
    if step >= max_steps:
        return "step_budget"
    if cost_spent >= max_cost:
        return "cost_budget"
    return None


FAULT_KINDS = ("malformed_args", "unknown_tool", "hallucinated_submit", "repeat_last")


class AgentTrace(NamedTuple):
    steps: list[dict]
    status: str          # "success" | "wrong_answer" | "step_budget" | "cost_budget" | "stuck"
    cost_spent: int
    n_steps: int


class ScriptedPolicy:
    """A deterministic, seeded stand-in for "the model" that proposes actions — an oracle
    that, at `fault_rate` per decision, substitutes a scripted mistake instead."""

    def __init__(self, plan: list[tuple[str, dict]], final_action: tuple[str, dict],
                fault_rate: float, rng: random.Random):
        self.plan = list(plan)
        self.final_action = final_action
        self.fault_rate = fault_rate
        self.rng = rng
        self.cursor = 0

    def _expected(self) -> tuple[str, dict]:
        return self.plan[self.cursor] if self.cursor < len(self.plan) else self.final_action

    def next_action(self, trace: list[dict]) -> tuple[str, dict]:
        if trace:
            last = trace[-1]
            if (self.cursor < len(self.plan) and last["ok"] and last["tool"] == "lookup"
                    and last["args"] == self.plan[self.cursor][1]):
                self.cursor += 1
        correct_tool, correct_args = self._expected()
        if self.rng.random() >= self.fault_rate:
            return correct_tool, dict(correct_args)
        fault = self.rng.choice(FAULT_KINDS)
        target = self.final_action[1]["value"]
        if fault == "malformed_args":
            return ("lookup", {}) if correct_tool == "lookup" else ("submit", {"value": str(target)})
        if fault == "unknown_tool":
            return "look_up", dict(correct_args)
        if fault == "hallucinated_submit":
            guess = target
            while guess == target:
                guess = self.rng.randint(0, 999)
            return "submit", {"value": guess}
        if trace:
            return trace[-1]["tool"], dict(trace[-1]["args"])
        return "lookup", {}


def run_agent_loop(policy: ScriptedPolicy, registry: ToolRegistry, max_steps: int,
                   max_cost: int, max_retries: int, stuck_window: int,
                   rng: random.Random, validate: bool = True) -> AgentTrace:
    """Run one episode: ask `policy` for actions, execute them through `registry`, and stop."""
    trace: list[dict] = []
    cost_spent = 0
    while True:
        status = budget_status(len(trace), cost_spent, max_steps, max_cost)
        if status is not None:
            return AgentTrace(steps=trace, status=status, cost_spent=cost_spent,
                              n_steps=len(trace))
        tool, args = policy.next_action(trace)
        result, attempts, _backoff = call_tool_with_retries(registry, tool, args, max_retries,
                                                             rng, validate=validate)
        cost_spent += attempts
        trace.append({"tool": tool, "args": args, "ok": result.ok, "error": result.error,
                     "attempts": attempts, "output": result.output})
        if tool == "submit" and result.ok:
            status = "success" if result.output.get("correct") else "wrong_answer"
            return AgentTrace(steps=trace, status=status, cost_spent=cost_spent,
                              n_steps=len(trace))
        if is_stuck(trace, stuck_window):
            return AgentTrace(steps=trace, status="stuck", cost_spent=cost_spent,
                              n_steps=len(trace))


_demo_task = generate_chain_task(length=3, seed=1)
print(f"a length-3 task: start at {_demo_task.start_key!r}, target {_demo_task.target}")

## 2. "It worked when I tried it" is not an evaluation

One episode of `run_agent_loop` tells you what happened once. A team shipping an agent needs
to know what happens ACROSS many tasks, and across many independent attempts at the SAME
task — because the same agent, on the same task, does not do the same thing twice: the
policy's own fault draws and the tool's own transient failures are both random. The rest of
this lesson builds the four things a defensible agent evaluation report needs: repeated,
seeded trials; the right reliability numbers, with the right definitions; trajectory and
failure evidence read from traces instead of by eye; and a comparison of two configurations
decided by an interval, not a single number that happened to be higher.

## 3. Exercise 1 — `run_suite_trials()`

Run `k` INDEPENDENT trials of every task in a suite. "Independent" is the graded part: each
trial needs its own tool registry, its own policy and its own retry randomness, seeded so
that (a) the whole suite is exactly reproducible from one `seed`, and (b) trial `t` of task
`i` never draws from the same random stream as trial `t'` of task `i`, or trial `t` of a
DIFFERENT task `i'` — otherwise your "k trials" are k copies of one trial.

<details><summary>💡 Hint 1 — what to think about</summary>

`evaluate_suite` in T07-L01 gives each task its own three `random.Random` streams (tool,
policy, retry), keyed by the seed and the task index. Here, build three from an integer
`base`, `base + 1`, `base + 2`. You need one more axis — the trial index `t` — folded into
`base` alongside `i`, spaced far enough apart that no `(task, trial)` pair's three RNGs can
ever collide with another pair's.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Two nested loops, task index `i` then trial index `t` in `range(k)`. For each pair compute
`base = seed * 1_000_000 + i * 1_000 + t * 4`, then `tool_rng = random.Random(base)`,
`policy_rng = random.Random(base + 1)`, `retry_rng = random.Random(base + 2)` — exactly
the same three roles T07-L01's `evaluate_suite` gives each task, from integer bases with a
trial term added. Build the
registry and the policy exactly as `evaluate_suite` did, run `run_agent_loop`, and collect
one list of `k` traces per task.
</details>

In [ ]:
def run_suite_trials(tasks: list[ChainTask], k: int, fault_rate: float, transient_rate: float,
                     max_steps: int, max_cost: int, max_retries: int, stuck_window: int,
                     validate: bool, seed: int) -> list[list[AgentTrace]]:
    """Run `k` independently seeded trials of every task in `tasks`.

    For task index `i` (0-based position in `tasks`) and trial index `t` (0-based, in
    `range(k)`), the trial's randomness comes from `base = seed * 1_000_000 + i * 1_000 +
    t * 4`: `tool_rng = random.Random(base)` (passed to `make_registry`), `policy_rng =
    random.Random(base + 1)` (passed to `ScriptedPolicy`), `retry_rng = random.Random(base
    + 2)` (passed to `run_agent_loop`) — one stream per role, as in T07-L01's `evaluate_suite`, but keyed
    by an integer base with a trial term added. Valid for up to 1000 tasks and up to 250 trials per task without two pairs' bases
    colliding.

    Returns a list the same length as `tasks`; entry `i` is a list of `k` `AgentTrace`
    objects, trial 0 first.

    Example:
        >>> tasks = generate_task_suite([2], n_per_length=1, seed=1)
        >>> trials = run_suite_trials(tasks, k=3, fault_rate=0.0, transient_rate=0.0,
        ...     max_steps=10, max_cost=10, max_retries=2, stuck_window=3, validate=True, seed=0)
        >>> len(trials), len(trials[0])
        (1, 3)
        >>> all(t.status == "success" for t in trials[0])
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_run_suite_trials() -> None:
    tasks = generate_task_suite([2, 3], n_per_length=1, seed=5)
    trials = run_suite_trials(tasks, k=4, fault_rate=0.2, transient_rate=0.15, max_steps=30,
                              max_cost=60, max_retries=3, stuck_window=3, validate=True, seed=7)
    assert len(trials) == 2 and all(len(t) == 4 for t in trials), (
        f"2 tasks with k=4 must give 2 lists of 4 traces, got shape "
        f"{[len(t) for t in trials]}"
    )
    again = run_suite_trials(tasks, k=4, fault_rate=0.2, transient_rate=0.15, max_steps=30,
                             max_cost=60, max_retries=3, stuck_window=3, validate=True, seed=7)
    assert [t.status for row in trials for t in row] == [t.status for row in again for t in row], (
        "the same seed must reproduce the same statuses every time"
    )

    def _reference(i: int, t: int, seed: int) -> AgentTrace:
        base = seed * 1_000_000 + i * 1_000 + t * 4
        task = tasks[i]
        registry = make_registry(task.chain, task.target, 0.15, random.Random(base))
        plan = [("lookup", {"key": key}) for key in task.key_sequence]
        policy = ScriptedPolicy(plan, ("submit", {"value": task.target}), 0.2,
                                random.Random(base + 1))
        return run_agent_loop(policy, registry, 30, 60, 3, 3, random.Random(base + 2),
                              validate=True)

    for i, t in ((0, 0), (0, 1), (1, 0), (1, 3)):
        expected = _reference(i, t, seed=7)
        got = trials[i][t]
        assert got == expected, (
            f"task {i} trial {t} does not match base = seed*1_000_000 + i*1_000 + t*4: "
            f"expected status={expected.status} n_steps={expected.n_steps} cost={expected.cost_spent}, "
            f"got status={got.status} n_steps={got.n_steps} cost={got.cost_spent} — check the "
            "exact seed formula and that tool_rng/policy_rng/retry_rng are base, base+1, base+2"
        )
    loose = run_suite_trials(tasks, k=3, fault_rate=0.35, transient_rate=0.3, max_steps=25,
                             max_cost=70, max_retries=1, stuck_window=4, validate=False, seed=3)
    for i, t in ((i, t) for i in range(2) for t in range(3)):
        base = 3 * 1_000_000 + i * 1_000 + t * 4
        task = tasks[i]
        policy = ScriptedPolicy([("lookup", {"key": key}) for key in task.key_sequence],
                                ("submit", {"value": task.target}), 0.35, random.Random(base + 1))
        expected = run_agent_loop(policy, make_registry(task.chain, task.target, 0.3,
                                                        random.Random(base)),
                                  25, 70, 1, 4, random.Random(base + 2), validate=False)
        assert loose[i][t] == expected, (
            f"with validate=False, max_retries=1 and stuck_window=4, task {i} trial {t} does not "
            "match — pass `validate` through to run_agent_loop instead of assuming True, and "
            "pass max_retries and stuck_window in that order (the other checks set both to 3, "
            "so a swap is invisible there)"
        )
    only_task_seeded = run_suite_trials(tasks[:1], k=4, fault_rate=0.2, transient_rate=0.15,
                                        max_steps=30, max_cost=60, max_retries=3, stuck_window=3,
                                        validate=True, seed=7)
    assert [t.status for t in only_task_seeded[0]] == [t.status for t in trials[0]], (
        "task 0's 4 trials must be identical whether or not task 1 is also in the suite — the "
        "seed for (task i, trial t) must not depend on how many OTHER tasks are in the list"
    )


_try("exercise 1", _check_run_suite_trials)

## 4. Exercise 2 — `task_success_counts()`

Every reliability number in this lesson starts here: for each task, how many of its `k`
trials ended `"success"`, out of how many trials. Keep this PER TASK — never pooled across
tasks — because every exercise from here on (the bootstrap, pass@k, pass^k, the paired
comparison) needs the per-task pair, not a single suite-wide total.

<details><summary>💡 Hint 1 — what to think about</summary>

One line of work per task: count how many of its trials have `status == "success"`, and
remember how many trials it had. The trial COUNT can differ from task to task if you ever
call this on trials from different runs — do not assume it is always `k`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

A list comprehension over `trials`, one `(successes, n)` tuple per task, in the SAME order
as `trials` — later code lines this array up against `tasks` by position.
</details>

In [ ]:
def task_success_counts(trials: list[list[AgentTrace]]) -> list[tuple[int, int]]:
    """Per-task `(successes, n)`, in the same order as `trials`.

    `successes` is how many of that task's trials have `status == "success"`; `n` is
    `len(trials[i])`.

    Example:
        >>> tasks = generate_task_suite([1], n_per_length=1, seed=2)
        >>> trials = run_suite_trials(tasks, k=4, fault_rate=0.0, transient_rate=0.0,
        ...     max_steps=5, max_cost=5, max_retries=1, stuck_window=3, validate=True, seed=0)
        >>> task_success_counts(trials)
        [(4, 4)]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _make_trace(status: str) -> AgentTrace:
    return AgentTrace(steps=[], status=status, cost_spent=1, n_steps=1)


def _check_task_success_counts() -> None:
    trials = [[_make_trace("success"), _make_trace("success"), _make_trace("stuck")],
             [_make_trace("wrong_answer"), _make_trace("wrong_answer")]]
    counts = task_success_counts(trials)
    assert counts == [(2, 3), (0, 2)], (
        f"got {counts!r}, expected [(2, 3), (0, 2)] — count successes and trials PER TASK, in "
        "the same order as `trials`, never pooled across tasks"
    )
    all_success = task_success_counts([[_make_trace("success")] * 5])
    assert all_success == [(5, 5)], f"all 5 trials succeed, expected [(5, 5)], got {all_success!r}"
    uneven = task_success_counts([[_make_trace("success")] * 3, [_make_trace("success")] * 7])
    assert [n for _, n in uneven] == [3, 7], (
        f"got n values {[n for _, n in uneven]!r} — do not assume every task has the same "
        "number of trials"
    )


_try("exercise 2", _check_task_success_counts)

## 5. Bootstrapping over tasks, not over trials

A percentile bootstrap resamples whatever you hand it, with replacement, and reads off the
middle 95% of the resampled means. What you hand it decides what question it answers. Given
below: the same mechanics you built in T07-L01, generalised to any array of numbers, one per
resampled UNIT — with one change. `numpy`'s own `.mean()` picks its summation method for
speed, and its documentation says the exact precision may vary; `_exact_means` below sums
each row with `math.fsum` instead, which avoids loss of precision by tracking several
partial sums (see claims.yaml), so this lesson prints the same numbers on Python 3.11 and 3.12.

In [ ]:
def _exact_means(rows: np.ndarray) -> np.ndarray:
    """Each row's mean, summed by `math.fsum` — never `.mean(axis=1)`, whose summation method
    (and so its last bit) is numpy's own choice, not this lesson's. Given."""
    return np.array([math.fsum(row) for row in rows]) / rows.shape[1]


def _percentile_bootstrap_mean(values: Sequence[float], n_boot: int,
                               seed: int) -> tuple[float, float, float]:
    """Percentile bootstrap CI for the mean of `values` — T07-L01's bootstrap mechanics,
    generalised from booleans to any float, with every mean summed exactly. Given."""
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = _exact_means(arr[idx])
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return float(math.fsum(arr) / n), float(lo), float(hi)


LENGTHS = [2, 4, 8, 12, 16]
N_PER_LENGTH = 8
K_TRIALS = 12
FAULT_RATE = 0.15
TRANSIENT_RATE = 0.12
MAX_RETRIES = 3
STUCK_WINDOW = 3
MAX_STEPS = 80
MAX_COST = 200
N_BOOT = 3000

_SUITE = generate_task_suite(LENGTHS, N_PER_LENGTH, seed=SEED)
print(f"suite: {len(_SUITE)} tasks over {len(LENGTHS)} lengths ({LENGTHS[0]}..{LENGTHS[-1]}), "
     f"{N_PER_LENGTH} tasks per length, {K_TRIALS} trials per task")

_TRIAL_CACHE: dict[str, list[list[AgentTrace]]] = {}


def _get_baseline_trials() -> list[list[AgentTrace]]:
    if "baseline" not in _TRIAL_CACHE:
        _TRIAL_CACHE["baseline"] = run_suite_trials(
            _SUITE, K_TRIALS, FAULT_RATE, TRANSIENT_RATE, MAX_STEPS, MAX_COST, MAX_RETRIES,
            STUCK_WINDOW, validate=True, seed=SEED)
    return _TRIAL_CACHE["baseline"]


def _show_bootstrap_over_tasks_vs_trials() -> None:
    trials = _get_baseline_trials()
    counts = task_success_counts(trials)
    task_rates = [c / n for c, n in counts]
    trial_outcomes = [tr.status == "success" for task_trials in trials for tr in task_trials]
    point_t, lo_t, hi_t = _percentile_bootstrap_mean(task_rates, N_BOOT, SEED)
    point_p, lo_p, hi_p = _percentile_bootstrap_mean(trial_outcomes, N_BOOT, SEED)
    width_t, width_p = hi_t - lo_t, hi_p - lo_p
    print(f"same suite, same {sum(n for _, n in counts)} trials, two different bootstraps:")
    print(f"  over the {len(counts)} TASKS   : {100 * point_t:.1f}%  "
         f"95% CI [{100 * lo_t:.1f}%, {100 * hi_t:.1f}%]  width {100 * width_t:.1f} points")
    print(f"  over the {len(trial_outcomes)} pooled TRIALS: {100 * point_p:.1f}%  "
         f"95% CI [{100 * lo_p:.1f}%, {100 * hi_p:.1f}%]  width {100 * width_p:.1f} points")
    narrower = "narrower" if width_p < width_t else "not narrower"
    print(f"  pooling over trials makes the interval {narrower} ({100 * abs(width_t - width_p):.1f} "
         "points) — it treats every trial as an independent unit, when trials of the SAME task "
         "share that task's own difficulty. The unit that varies independently across the suite "
         "is the TASK; that is what a bootstrap must resample.")


_try("bootstrap: tasks vs trials", _show_bootstrap_over_tasks_vs_trials,
     needs=("exercise 1", "exercise 2"))

## 6. Exercise 3 — `pass_at_k()`

The code-generation literature's answer to "did the model solve it": run `n >= k` trials,
and ask whether AT LEAST ONE of `k` of them succeeds. Estimating that by running exactly `k`
trials and checking whether any succeeded has high variance; the Codex/HumanEval paper's fix
is an unbiased estimator that uses ALL `n` trials (see claims.yaml).

<details><summary>💡 Hint 1 — what to think about</summary>

For one task with `n` trials and `c` successes, `pass@k := 1 - C(n-c, k) / C(n, k)`: the
chance that a random k-subset of the `n` trials is NOT entirely failures. When `n - c < k`
there are not `k` failures to choose, so `C(n-c, k)` is 0 — what must `pass@k` be in that
case, without computing anything?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

The Codex paper's own concern is numerical: computing the estimator directly "results in
very large numbers and numerical instability", so its numpy code evaluates the ratio as a
termwise product instead (claims.yaml). Python's integers do not have that problem — `int`
is arbitrary-precision, so `math.comb(n, k)` (standard library, no import beyond `math`) is
always exact, however large `n` gets. Compute the RATIO of two exact integers from `math.comb`
directly, with one `/`. Average the per-task estimate over every task in `counts` with
`math.fsum(estimates) / len(estimates)` — never the built-in `sum()`, whose float summation
changed in Python 3.12 (see section 5 and claims.yaml).
</details>

In [ ]:
def pass_at_k(counts: list[tuple[int, int]], k: int) -> float:
    """The Codex/HumanEval unbiased pass@k estimator, averaged over tasks.

    Each `(c, n)` in `counts` needs `n >= k`; raises `ValueError` naming the shortfall
    otherwise. Per task: `1.0` if `n - c < k` (fewer than `k` failures exist, so any k-subset
    must include a success); otherwise `1.0 - math.comb(n - c, k) / math.comb(n, k)`. Returns
    `math.fsum(estimates) / len(estimates)` — Python's `int` is arbitrary-precision, so
    `math.comb` never produces the "very large numbers" the Codex paper's numpy code works
    around, and `math.fsum` (rather than the built-in `sum`) gives the same average on every
    Python version, the same discipline section 5's bootstrap uses.

    Example:
        >>> round(pass_at_k([(2, 4)], k=2), 6)
        0.833333
        >>> pass_at_k([(5, 5)], k=1)
        1.0
        >>> pass_at_k([(0, 5)], k=1)
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pass_at_k() -> None:
    exact = pass_at_k([(2, 4)], k=2)
    assert abs(exact - 5 / 6) < 1e-9, (
        f"C(2,2)/C(4,2) = 1/6, so pass@2 with c=2,n=4 must be 1 - 1/6 = 5/6, got {exact}"
    )
    full = pass_at_k([(1, 5)], k=5)
    assert full == 1.0, (
        f"k == n means every trial is in the one possible 'subset', and c=1 success exists, "
        f"so pass@k must be exactly 1.0, got {full} — check the n - c < k branch"
    )
    near_full = pass_at_k([(1, 5)], k=4)
    assert abs(near_full - 0.8) < 1e-9, (
        f"c=1, n=5, k=4: P(miss the one success) = C(4,4)/C(5,4) = 1/5, so pass@4 = 0.8, got "
        f"{near_full} — this is the k=5 case above with k reduced by exactly one"
    )
    zero = pass_at_k([(0, 5)], k=3)
    assert zero == 0.0, f"0 successes out of 5 trials: no subset can contain one, got {zero}"
    averaged = pass_at_k([(4, 4), (1, 8)], k=1)
    assert abs(averaged - 0.5625) < 1e-9, (
        f"got {averaged} for two tasks with pass@1 = 4/4 and 1/8 — average the PER-TASK "
        "estimates (0.5625), do not pool the two tasks' counts into one c and n (5/12)"
    )
    monotone = [pass_at_k([(6, 10), (2, 10)], k=k) for k in (1, 2, 4, 6, 8, 10)]
    assert all(a <= b + 1e-9 for a, b in zip(monotone, monotone[1:])), (
        f"pass@k must never DECREASE as k grows (more trials can only help "
        f"'at least one succeeds'), got {monotone}"
    )
    try:
        pass_at_k([(3, 5)], k=6)
        raise AssertionError("k=6 with only n=5 trials must raise ValueError, not return a number")
    except ValueError:
        pass


_try("exercise 3", _check_pass_at_k)

## 7. Exercise 4 — `pass_hat_k()`

τ-bench's answer to a different question: not "did at least one of k trials succeed" but
"would ALL k succeed" — the chance a random k-subset of the `n` trials is entirely
successes. The paper's own name for it is pass^k ("pass hat k"); see claims.yaml for its
definition, verbatim.

<details><summary>💡 Hint 1 — what to think about</summary>

`pass^k := C(c, k) / C(n, k)`: the chance a random k-subset of the `n` trials is made ENTIRELY
of the `c` successes. If `c < k` there are not even `k` successes to choose from — what must
`pass^k` be then, without computing anything?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

The same ratio as pass@k, the other way round: `pass^k = C(c, k) / C(n, k)`. `math.comb`
again — exact for any `c`, `n`, no overflow, no termwise product to accumulate rounding
error in. Average the per-task estimate with `math.fsum(estimates) / len(estimates)`, exactly
like pass@k.
</details>

In [ ]:
def pass_hat_k(counts: list[tuple[int, int]], k: int) -> float:
    """τ-bench's pass^k: the chance ALL of a random k-subset of trials succeed, averaged over
    tasks.

    Each `(c, n)` in `counts` needs `n >= k`; raises `ValueError` naming the shortfall
    otherwise. Per task: `0.0` if `c < k` (fewer than `k` successes exist); otherwise
    `math.comb(c, k) / math.comb(n, k)`, the exact ratio of two `math.comb` integers. Returns
    `math.fsum(estimates) / len(estimates)`, exactly like pass@k.

    Example:
        >>> round(pass_hat_k([(2, 4)], k=2), 6)
        0.166667
        >>> pass_hat_k([(3, 5)], k=3)
        0.1
        >>> pass_hat_k([(2, 5)], k=3)
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pass_hat_k() -> None:
    exact = pass_hat_k([(2, 4)], k=2)
    assert abs(exact - 1 / 6) < 1e-9, (
        f"C(2,2)/C(4,2) = 1/6, so pass^2 with c=2,n=4 must be 1/6, got {exact}"
    )
    boundary_hit = pass_hat_k([(3, 5)], k=3)
    assert abs(boundary_hit - 0.1) < 1e-9, (
        f"c == k == 3: C(3,3)/C(5,3) = 1/10, got {boundary_hit} — check the c < k branch is "
        "not firing when c equals k"
    )
    boundary_miss = pass_hat_k([(2, 5)], k=3)
    assert boundary_miss == 0.0, (
        f"c=2 is one short of k=3: no 3-subset can be all successes, got {boundary_miss}"
    )
    full = pass_hat_k([(5, 5)], k=5)
    assert full == 1.0, f"c == n == k means the only subset IS all successes, got {full}"
    averaged = pass_hat_k([(4, 4), (1, 8)], k=1)
    assert abs(averaged - 0.5625) < 1e-9, (
        f"got {averaged} for two tasks with pass^1 = 4/4 and 1/8 — average the PER-TASK "
        "estimates (0.5625), do not pool the two tasks' counts into one c and n (5/12)"
    )
    monotone = [pass_hat_k([(6, 10), (2, 10)], k=k) for k in (1, 2, 4, 6, 8, 10)]
    assert all(a >= b - 1e-9 for a, b in zip(monotone, monotone[1:])), (
        f"pass^k must never INCREASE as k grows (requiring more trials to ALL succeed can "
        f"only get harder), got {monotone}"
    )
    try:
        pass_hat_k([(3, 5)], k=6)
        raise AssertionError("k=6 with only n=5 trials must raise ValueError, not return a number")
    except ValueError:
        pass


_try("exercise 4", _check_pass_hat_k)

## 8. pass@k rises, pass^k falls — on this lesson's own numbers

Nothing below is asserted; every number is computed from the baseline suite's trials.

In [ ]:
def _show_pass_at_and_hat_k() -> None:
    counts = task_success_counts(_get_baseline_trials())
    ks = [k for k in (1, 3, 6, 12) if k <= K_TRIALS]
    print(f"{'k':>3}  {'pass@k':>8}  {'pass^k':>8}")
    at_row, hat_row = [], []
    for k in ks:
        a, h = pass_at_k(counts, k), pass_hat_k(counts, k)
        at_row.append(a)
        hat_row.append(h)
        print(f"{k:>3}  {100 * a:7.1f}%  {100 * h:7.1f}%")
    at_trend = "rose" if at_row[-1] > at_row[0] else "did not rise"
    hat_trend = "fell" if hat_row[-1] < hat_row[0] else "did not fall"
    print(f"\nfrom k={ks[0]} to k={ks[-1]}: pass@k {at_trend}, from {100 * at_row[0]:.1f}% to "
         f"{100 * at_row[-1]:.1f}%; pass^k {hat_trend}, from {100 * hat_row[0]:.1f}% to "
         f"{100 * hat_row[-1]:.1f}%.")
    print("Both are computed from the SAME trials. pass@k answers 'could it eventually get "
         "there' and grows the more attempts you allow it; pass^k answers 'would it get there "
         "reliably, every time' and shrinks for exactly the same reason. A team that reports "
         "pass@k as the agent's reliability is reporting the first question while meaning to "
         "ask the second, and the gap between the two rows above is how far off that report "
         "would be.")


_try("pass@k vs pass^k demo", _show_pass_at_and_hat_k,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 9. Exercise 5 — `trajectory_metrics()`

A success rate says nothing about HOW an episode got there. Every trial's `AgentTrace` has
already recorded every step; this exercise reads the trajectory evidence out of it: how long
episodes run, how often a tool call fails, how often the agent carries on afterwards instead
of the episode ending right there, and what a success costs once every failed attempt is
counted in.

<details><summary>💡 Hint 1 — what to think about</summary>

A "tool error" is any step with `ok=False`, wherever it falls in its trace. A "recovery" is
narrower: a failed step that is NOT the last entry of ITS trace — because if the loop
recorded another step afterwards, the agent's next turn treated the failure as something to
act on, not the end of the episode.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Flatten nothing conceptually: loop over every trace, and within each trace, over
`enumerate(trace.steps)`. Count `ok=False` steps for `tool_errors`; count `ok=False` steps
whose index is less than `len(trace.steps) - 1` for `recoveries`. `mean_steps` is the mean of
every trace's `n_steps`; `cost_per_success` is the sum of EVERY trace's `cost_spent`
(successes and failures alike) divided by how many traces succeeded — `inf` if none did.
</details>

In [ ]:
class TrajectoryMetrics(NamedTuple):
    n_episodes: int
    mean_steps: float
    tool_errors: int
    recoveries: int
    cost_per_success: float


def trajectory_metrics(traces: list[AgentTrace]) -> TrajectoryMetrics:
    """Aggregate trajectory evidence read directly from a flat list of traces.

    `tool_errors` counts every step with `ok=False` across every trace. `recoveries` counts
    the ones that are NOT their trace's last step — the loop recorded a further step
    afterwards, so the agent acted again rather than the episode ending there.
    `cost_per_success` is `sum(trace.cost_spent for trace in traces)` (every trace, not only
    the successful ones) divided by the number of traces with `status == "success"`; `inf`
    when there are none.

    Example:
        >>> ok = {"tool": "lookup", "args": {}, "ok": True, "error": None, "output": None, "attempts": 1}
        >>> bad = {"tool": "lookup", "args": {}, "ok": False, "error": "x", "output": None, "attempts": 1}
        >>> t1 = AgentTrace(steps=[bad, ok], status="success", cost_spent=2, n_steps=2)
        >>> t2 = AgentTrace(steps=[bad], status="stuck", cost_spent=1, n_steps=1)
        >>> m = trajectory_metrics([t1, t2])
        >>> m.n_episodes, m.tool_errors, m.recoveries
        (2, 2, 1)
        >>> m.cost_per_success
        3.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_trajectory_metrics() -> None:
    ok = {"tool": "lookup", "args": {}, "ok": True, "error": None, "output": None, "attempts": 1}
    bad = {"tool": "lookup", "args": {}, "ok": False, "error": "x", "output": None, "attempts": 1}
    t1 = AgentTrace(steps=[bad, ok], status="success", cost_spent=2, n_steps=2)
    t2 = AgentTrace(steps=[bad], status="stuck", cost_spent=1, n_steps=1)
    m = trajectory_metrics([t1, t2])
    assert m.n_episodes == 2, f"got n_episodes={m.n_episodes}, expected 2"
    assert m.tool_errors == 2, (
        f"got tool_errors={m.tool_errors}, expected 2 — one failed step in each trace, "
        "wherever it falls"
    )
    assert m.recoveries == 1, (
        f"got recoveries={m.recoveries}, expected 1 — t1's failed step is followed by another "
        "step (recovered); t2's failed step is its trace's LAST (and only) step, so it is not "
        "a recovery"
    )
    assert abs(m.mean_steps - 1.5) < 1e-9, f"mean of n_steps 2 and 1 is 1.5, got {m.mean_steps}"
    assert abs(m.cost_per_success - 3.0) < 1e-9, (
        f"got cost_per_success={m.cost_per_success}, expected 3.0 (costs 2 + 1, ONE success) "
        "— total cost across every trace, not only the successful ones, divided by successes"
    )
    only_last_fails = AgentTrace(steps=[ok, ok, bad], status="stuck", cost_spent=3, n_steps=3)
    m2 = trajectory_metrics([only_last_fails])
    assert m2.tool_errors == 1 and m2.recoveries == 0, (
        f"a trace whose ONLY failure is its last step has 1 tool_error and 0 recoveries, got "
        f"tool_errors={m2.tool_errors} recoveries={m2.recoveries}"
    )
    back_to_back = trajectory_metrics([AgentTrace(steps=[bad, bad, ok], status="success",
                                                  cost_spent=3, n_steps=3)])
    assert (back_to_back.tool_errors, back_to_back.recoveries) == (2, 2), (
        f"[failed, failed, ok] has 2 tool_errors and 2 recoveries, got "
        f"{(back_to_back.tool_errors, back_to_back.recoveries)} — a failed step is a recovery "
        "whenever ANY step follows it, even another failure, not only a successful one"
    )
    no_success = trajectory_metrics([AgentTrace(steps=[bad], status="stuck", cost_spent=1, n_steps=1)])
    assert no_success.cost_per_success == float("inf"), (
        f"zero successes must give cost_per_success = inf, got {no_success.cost_per_success}"
    )


_try("exercise 5", _check_trajectory_metrics)

## 10. Exercise 6 — `failure_taxonomy_from_traces()`

The status field on every failed trace, tallied — computed from the traces themselves, not
eyeballed from a printout.

<details><summary>💡 Hint 1 — what to think about</summary>

Every `AgentTrace.status` that is not `"success"` is a failure kind. Count them; a
`dict` remembers the order its keys were first added, and Python's `sorted` is stable, so
sorting by count descending leaves equal counts in the order you first met them.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

One pass over `traces`, skipping `status == "success"`, building a `{status: count}` dict.
Return it sorted by count, highest first.
</details>

In [ ]:
def failure_taxonomy_from_traces(traces: list[AgentTrace]) -> dict[str, int]:
    """Counts of each non-success `status` across `traces`, highest count first.

    Example:
        >>> failure_taxonomy_from_traces([
        ...     AgentTrace([], "wrong_answer", 1, 1), AgentTrace([], "stuck", 1, 3),
        ...     AgentTrace([], "wrong_answer", 1, 1), AgentTrace([], "success", 1, 2)])
        {'wrong_answer': 2, 'stuck': 1}
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_failure_taxonomy_from_traces() -> None:
    traces = [_make_trace("wrong_answer"), _make_trace("stuck"), _make_trace("wrong_answer"),
             _make_trace("success"), _make_trace("cost_budget")]
    tax = failure_taxonomy_from_traces(traces)
    assert tax == {"wrong_answer": 2, "stuck": 1, "cost_budget": 1}, (
        f"got {tax!r}, expected {{'wrong_answer': 2, 'stuck': 1, 'cost_budget': 1}} — exclude "
        "successes, sort by count descending, and keep first-seen order among ties"
    )
    rarest_first = failure_taxonomy_from_traces(
        [_make_trace("stuck"), _make_trace("cost_budget"), _make_trace("cost_budget")])
    assert list(rarest_first.items()) == [("cost_budget", 2), ("stuck", 1)], (
        f"got {list(rarest_first.items())!r} — 'stuck' was seen first but occurs once; sort by "
        "count, highest first (first-seen order only breaks ties)"
    )
    empty = failure_taxonomy_from_traces([_make_trace("success")] * 3)
    assert empty == {}, f"all successes must give an empty taxonomy, got {empty!r}"
    tie_order = failure_taxonomy_from_traces([_make_trace("stuck"), _make_trace("cost_budget")])
    assert list(tie_order) == ["stuck", "cost_budget"], (
        f"got key order {list(tie_order)!r} — with a tied count (1 each), 'stuck' was seen "
        "first and must come first"
    )


_try("exercise 6", _check_failure_taxonomy_from_traces)

Trajectory metrics and the failure taxonomy on the baseline suite's trials, flattened once:

In [ ]:
def _show_baseline_trajectory() -> None:
    flat = [tr for task_trials in _get_baseline_trials() for tr in task_trials]
    m = trajectory_metrics(flat)
    print(f"{m.n_episodes} episodes  ·  mean steps {m.mean_steps:.2f}  ·  "
         f"tool errors {m.tool_errors}  ·  recoveries {m.recoveries}  ·  "
         f"cost/success {m.cost_per_success:.2f}")
    recovered = "recovered from" if m.recoveries else "never recovered from"
    print(f"this agent {recovered} a tool error and carried on, "
         f"{m.recoveries} of {m.tool_errors} times")
    hit = [tr for tr in flat if any(not step["ok"] for step in tr.steps)]
    reached = sum(tr.status == "success" for tr in hit)
    print(f"but carrying on is not getting there: of the {len(hit)} episodes that hit a tool "
         f"error, {reached} ended in success ({100 * reached / max(len(hit), 1):.1f}%)")
    print("failure taxonomy:", failure_taxonomy_from_traces(flat))


_try("baseline trajectory demo", _show_baseline_trajectory,
     needs=("exercise 1", "exercise 5", "exercise 6"))

## 11. Exercise 7 — `paired_diff_interval()`

Comparing two configurations by their two separate success rates throws away the one thing
that makes a small difference detectable: the two rates were measured on the SAME tasks, so
task-to-task difficulty should not count as noise. A PAIRED bootstrap keeps that pairing
intact: every resample draws the same task indices for both sides, so a resample that happens
to include an easy task includes it for BOTH configurations, and only the actual difference
between them survives.

<details><summary>💡 Hint 1 — what to think about</summary>

`counts_a` and `counts_b` must describe the SAME tasks, in the same order — enforce
`len(counts_a) == len(counts_b)` up front. Turn each into a per-task rate array. One draw of
resample indices, reused for BOTH arrays, is what makes this "paired" rather than two
separate bootstraps.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

`rng.integers(0, n_tasks, size=(n_boot, n_tasks))` gives one grid of resampled task indices;
index BOTH rate arrays with it and reduce each row with `_exact_means` (section 5's helper),
then subtract (b's resampled mean minus a's, per row) to get `n_boot` resampled differences.
The point estimate is `math.fsum(rate_b) / n - math.fsum(rate_a) / n` on the ORIGINAL data,
never the mean of the resampled differences. The interval is the 2.5th/97.5th percentiles of
the resampled differences.
</details>

In [ ]:
def paired_diff_interval(counts_a: list[tuple[int, int]], counts_b: list[tuple[int, int]],
                         n_boot: int, seed: int) -> tuple[float, float, float]:
    """Paired percentile bootstrap CI for the mean per-task difference `rate_b - rate_a`.

    `counts_a[i]` and `counts_b[i]` must describe the SAME task `i`; raises `ValueError` if
    `len(counts_a) != len(counts_b)`. Converts each to a per-task success rate (`c / n`), draws
    ONE grid of task-index resamples with `numpy.random.default_rng(seed).integers(0, n_tasks,
    size=(n_boot, n_tasks))`, and applies it to BOTH rate arrays — never two independent
    resamplings — before differencing, reducing each resampled row with `_exact_means` rather
    than `.mean(axis=1)`. Returns `(point, lo, hi)`: `point` is `mean(rate_b) - mean(rate_a)` on
    the original data, computed with `math.fsum` for the same exact-reproducibility reason as
    section 5's bootstrap; `lo`/`hi` are the 2.5th/97.5th percentiles of the `n_boot` resampled
    differences.

    Example:
        >>> a = [(5, 10), (6, 10), (7, 10), (8, 10)]
        >>> b = [(6, 10), (7, 10), (8, 10), (9, 10)]
        >>> point, lo, hi = paired_diff_interval(a, b, n_boot=2000, seed=0)
        >>> round(point, 6)
        0.1
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_paired_diff_interval() -> None:
    a = [(5, 10), (6, 10), (7, 10), (8, 10)]
    b = [(6, 10), (7, 10), (8, 10), (9, 10)]     # every task exactly +0.1 over `a`
    point, lo, hi = paired_diff_interval(a, b, n_boot=4000, seed=0)
    assert abs(point - 0.1) < 1e-9, f"mean(b) - mean(a) must be exactly 0.1, got {point}"
    assert hi - lo < 0.02, (
        f"every task improves by EXACTLY the same 0.1, so a PAIRED resample must see the same "
        f"+0.1 difference on every single resample (interval width ~0) — got width "
        f"{hi - lo:.4f}, which means `a` and `b` were resampled with two INDEPENDENT index "
        "draws instead of one shared grid"
    )
    mixed_a = [(1, 10), (8, 10), (4, 10), (6, 10), (3, 10)]
    mixed_b = [(4, 10), (7, 10), (8, 10), (6, 10), (5, 10)]    # +0.3, -0.1, +0.4, 0, +0.2
    mixed, lo3, hi3 = paired_diff_interval(mixed_a, mixed_b, n_boot=2000, seed=3)
    assert abs(mixed - 0.16) < 1e-9, (
        f"mean(rate_b) - mean(rate_a) on the original data is 0.16, got {mixed} — the point is "
        "not the mean of the resampled differences"
    )
    assert lo3 < mixed < hi3, (
        f"got lo={lo3} point={mixed} hi={hi3}: when per-task differences vary, a 95% interval "
        "must straddle its own point — resample b minus a (the same way round as the point), "
        "lo at the 2.5th percentile, hi at the 97.5th"
    )
    grid = np.random.default_rng(3).integers(0, 5, size=(2000, 5))
    rates_a = np.array([c / n for c, n in mixed_a])
    rates_b = np.array([c / n for c, n in mixed_b])
    ref_lo, ref_hi = np.percentile(_exact_means(rates_b[grid]) - _exact_means(rates_a[grid]),
                                   [2.5, 97.5])
    assert abs(lo3 - ref_lo) < 1e-12 and abs(hi3 - ref_hi) < 1e-12, (
        f"got [{lo3}, {hi3}], expected [{ref_lo}, {ref_hi}] — use exactly the docstring's grid, "
        "default_rng(seed).integers(0, n_tasks, size=(n_boot, n_tasks)), for BOTH sides, and "
        "the 2.5th/97.5th percentiles (a 95% interval, not 90%)"
    )
    same, lo2, hi2 = paired_diff_interval(a, a, n_boot=1000, seed=1)
    assert same == 0.0 and lo2 == 0.0 and hi2 == 0.0, (
        f"a configuration compared against itself has NO difference on ANY resample, got "
        f"point={same} lo={lo2} hi={hi2}"
    )
    correctness_point, _, _ = paired_diff_interval([(3, 10)], [(9, 10)], n_boot=10, seed=2)
    assert abs(correctness_point - 0.6) < 1e-9, (
        f"one task, rate_a=0.3, rate_b=0.9: point must be exactly 0.6, got {correctness_point}"
    )
    try:
        paired_diff_interval([(3, 10), (4, 10)], [(3, 10)], n_boot=100, seed=0)
        raise AssertionError("mismatched task counts must raise ValueError, not silently truncate")
    except ValueError:
        pass


_try("exercise 7", _check_paired_diff_interval)

## 12. Exercise 8 — `compare_configurations()`

The whole pipeline, wired together correctly: run two agent CONFIGURATIONS — e.g. with and
without argument validation — on the SAME task suite and the SAME seed, so that each trial
starts from the same random streams (which fault the policy draws, when a tool times out) and
the two runs only drift apart where the one thing you changed makes them act differently. Get
the seed or the task list out of step between the two sides and you are no longer measuring
the configuration change; you are measuring noise. This is the mistake `compare_configurations(tasks, k, seed, cfg, cfg,
n_boot)` — the SAME configuration on both sides — is built to expose: done correctly, it
reports NO difference, because there is none to find.

<details><summary>💡 Hint 1 — what to think about</summary>

`config_a` and `config_b` are dicts of the keyword arguments `run_suite_trials` still needs
beyond `tasks`, `k` and `seed` (`fault_rate`, `transient_rate`, `max_steps`, `max_cost`,
`max_retries`, `stuck_window`, `validate`). Call `run_suite_trials` twice, passing `tasks`
and `seed` UNCHANGED both times and `**config_a` / `**config_b` for the rest.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Run the suite once per configuration — the same tasks, `k` and seed both times, only the
configuration dict differs. Turn each run into per-task counts with exercise 2, hand both
lists to exercise 7 with the `n_boot` and seed you were given, and work out each side's own
mean per-task rate for the printout with `math.fsum`, not the built-in `sum()`.
</details>

In [ ]:
class ComparisonResult(NamedTuple):
    point: float
    lo: float
    hi: float
    rate_a: float
    rate_b: float


def compare_configurations(tasks: list[ChainTask], k: int, seed: int, config_a: dict,
                           config_b: dict, n_boot: int) -> ComparisonResult:
    """Run two configurations on the SAME tasks and seed, and the paired difference between them.

    `config_a` / `config_b` supply `run_suite_trials`'s remaining keyword arguments
    (`fault_rate`, `transient_rate`, `max_steps`, `max_cost`, `max_retries`, `stuck_window`,
    `validate`); `tasks`, `k` and `seed` are passed to BOTH calls unchanged. Returns
    `ComparisonResult(point, lo, hi, rate_a, rate_b)`: `(point, lo, hi)` is
    `paired_diff_interval(counts_a, counts_b, n_boot, seed)` on the two runs' per-task counts
    (the same `seed` again), and `rate_a` / `rate_b` are each side's mean per-task success
    rate, averaged with `math.fsum`.

    Example:
        >>> tasks = generate_task_suite([2, 4], n_per_length=2, seed=1)
        >>> cfg = dict(fault_rate=0.1, transient_rate=0.1, max_steps=40, max_cost=80,
        ...            max_retries=3, stuck_window=3, validate=True)
        >>> same = compare_configurations(tasks, k=5, seed=0, config_a=cfg, config_b=cfg, n_boot=500)
        >>> same.point, same.lo, same.hi
        (0.0, 0.0, 0.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_compare_configurations() -> None:
    tasks = generate_task_suite([2, 4, 8], n_per_length=3, seed=11)
    base_cfg = dict(fault_rate=0.15, transient_rate=0.15, max_steps=40, max_cost=80,
                    max_retries=3, stuck_window=3, validate=True)
    same = compare_configurations(tasks, k=6, seed=3, config_a=base_cfg, config_b=base_cfg,
                                  n_boot=500)
    assert (same.point, same.lo, same.hi) == (0.0, 0.0, 0.0), (
        f"got point={same.point} lo={same.lo} hi={same.hi} for a configuration compared "
        "against ITSELF — this must be exactly zero. If it is not, the two run_suite_trials "
        "calls did not use the SAME seed and the SAME tasks (or config_a and config_b were "
        "mutated between the two calls)"
    )
    assert abs(same.rate_a - same.rate_b) < 1e-12, (
        f"rate_a ({same.rate_a}) and rate_b ({same.rate_b}) must match exactly for identical "
        "configurations"
    )
    no_validate_cfg = {**base_cfg, "validate": False}
    result = compare_configurations(tasks, k=6, seed=3, config_a=base_cfg,
                                    config_b=no_validate_cfg, n_boot=500)
    trials_a_direct = run_suite_trials(tasks, 6, seed=3, **base_cfg)
    counts_a_direct = task_success_counts(trials_a_direct)
    rate_a_direct = math.fsum(c / n for c, n in counts_a_direct) / len(counts_a_direct)
    assert abs(result.rate_a - rate_a_direct) < 1e-12, (
        f"got rate_a={result.rate_a}, expected {rate_a_direct} from calling run_suite_trials "
        "with config_a and the SAME seed directly — compare_configurations must not perturb "
        "the seed it was given"
    )
    assert abs(result.point - (result.rate_b - result.rate_a)) < 1e-9, (
        f"point ({result.point}) must equal rate_b - rate_a ({result.rate_b - result.rate_a})"
    )
    counts_b_direct = task_success_counts(run_suite_trials(tasks, 6, seed=3, **no_validate_cfg))
    rate_b_direct = math.fsum(c / n for c, n in counts_b_direct) / len(counts_b_direct)
    assert abs(result.rate_b - rate_b_direct) < 1e-12, (
        f"got rate_b={result.rate_b}, expected {rate_b_direct} from running config_b directly — "
        "the second run must use config_b, not config_a again"
    )
    _, lo_direct, hi_direct = paired_diff_interval(counts_a_direct, counts_b_direct, 500, 3)
    assert (result.lo, result.hi) == (lo_direct, hi_direct), (
        f"got [{result.lo}, {result.hi}], expected [{lo_direct}, {hi_direct}] from "
        "paired_diff_interval(counts_a, counts_b, n_boot, seed) — bootstrap with the n_boot and "
        "seed you were given, a's counts first"
    )


_try("exercise 8", _check_compare_configurations)

The paired comparison this lesson set out to build: validation on versus off, same tasks,
same seed, decided by an interval, not by which single number is larger.

In [ ]:
def _show_validation_comparison() -> None:
    base_cfg = dict(fault_rate=FAULT_RATE, transient_rate=TRANSIENT_RATE, max_steps=MAX_STEPS,
                    max_cost=MAX_COST, max_retries=MAX_RETRIES, stuck_window=STUCK_WINDOW,
                    validate=True)
    no_validate_cfg = {**base_cfg, "validate": False}
    result = compare_configurations(_SUITE, K_TRIALS, SEED, base_cfg, no_validate_cfg, N_BOOT)
    print(f"validation ON : mean task success rate {100 * result.rate_a:.1f}%")
    print(f"validation OFF: mean task success rate {100 * result.rate_b:.1f}%")
    print(f"paired difference (off - on): {100 * result.point:+.1f} points, "
         f"95% CI [{100 * result.lo:+.1f}, {100 * result.hi:+.1f}]")
    excludes_zero = result.lo > 0.0 or result.hi < 0.0
    verdict = ("does not include zero" if excludes_zero else "includes zero")
    direction = ("worse" if result.point < 0 else "better" if result.point > 0 else "no different")
    print(f"the interval {verdict}: removing validation measures as {direction}, and that "
         f"reading is {'' if excludes_zero else 'NOT '}distinguishable from no difference at "
         "all, on this suite and this many trials.")


_try("validation on vs off", _show_validation_comparison,
     needs=("exercise 1", "exercise 2", "exercise 7", "exercise 8"))

## 13. Common mistakes

- **Bootstrapping the pooled trial outcomes instead of the per-task rates.** Section 5's
  demo measures exactly how much narrower — and how wrong — that interval looks.
- **Reporting pass@k as "the" reliability number.** It answers "could it eventually solve
  this", not "would it solve this reliably"; pass^k answers the second question, and section
  8 shows the two moving in opposite directions on the same data.
- **Resampling `a` and `b` independently in a paired comparison.** That throws away the
  pairing, and when a task's difficulty moves both sides together — as it does here — it
  inflates the interval. Exercise 7's shared-shift check is built to catch exactly this.
- **Comparing two configurations run with different seeds, or on different task lists.**
  `compare_configurations(tasks, k, seed, cfg, cfg, n_boot)` must come back exactly zero;
  if it does not, the comparison is not paired at all.
- **Counting a "recovery" the same as a "tool error".** A failed step the loop continued
  past is a recovery; a failed step that WAS the episode's last word is not.

In [ ]:
def _show_same_record_two_numbers() -> None:
    demo_counts = [(3, 5)]
    print(f"pass@5 with a 3/5 record: {100 * pass_at_k(demo_counts, 5):.0f}% (at least one of "
         f"all 5 succeeds) vs pass^5: {100 * pass_hat_k(demo_counts, 5):.0f}% (all 5 succeed) "
         "-- the same 3/5 record, two very different numbers, which is exactly why reporting "
         "one when you mean the other is not a rounding error.")


_try("same record, two numbers", _show_same_record_two_numbers,
     needs=("exercise 3", "exercise 4"))

## 14. Self-check

1. A team reports "reliability is 92%, computed as pass@5 over 5 trials per task." What does
   that number actually show?
   - (a) The agent succeeds reliably, every time, on 92% of tasks
   - (b) At least one of 5 attempts succeeded on 92% of tasks — a much easier bar
   - (c) The agent succeeds with probability 0.92 on any single attempt
   - (d) 92% is a 95% confidence interval's lower bound

2. Why resample TASKS rather than pooling every trial into one array?
   - (a) It is computationally cheaper
   - (b) Trials of one task share that task's difficulty; pooling treats them as independent
     and understates the true uncertainty
   - (c) numpy cannot resample an array larger than the task count
   - (d) The two bootstraps always give an identical interval

3. `paired_diff_interval` applies ONE grid of resampled task indices to both `rate_a` and
   `rate_b`. What would two SEPARATE, independent grids do instead?
   - (a) Nothing; the two approaches are equivalent
   - (b) Narrow the interval, understating the true uncertainty
   - (c) Widen it, by mixing pairs of tasks that were never actually run together
   - (d) Change the point estimate, not just the interval

4. `compare_configurations(tasks, k, seed, cfg, cfg, n_boot)` — the SAME `cfg` as both
   sides — comes back with a non-zero point estimate. What does that tell you?
   - (a) The two configurations are subtly different after all
   - (b) Nothing is wrong; a little noise between two runs is expected
   - (c) The two calls inside it did not share `tasks` and `seed` — it is not paired
   - (d) `n_boot` was set too low

5. A trace's `status` is `"stuck"` and its LAST step has `ok=False`. Is that step a
   "recovery" in `trajectory_metrics`?
   - (a) Yes — the agent recovered from earlier failures too
   - (b) No — a recovery is a failed step that is NOT the trace's last step
   - (c) Yes — `is_stuck` is itself a form of recovery
   - (d) It depends on how many steps came before it

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
             "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built

`run_suite_trials` and `task_success_counts` turn one episode into a measurement. `pass_at_k`
and `pass_hat_k` are two different, precisely sourced answers to "is it reliable", and
section 8 shows why reporting the wrong one is not a rounding error. `trajectory_metrics` and
`failure_taxonomy_from_traces` turn a pile of traces into the evidence behind a success rate.
`paired_diff_interval` and `compare_configurations` are how a change to the agent gets judged:
not by which single number is bigger, but by whether an interval on the paired difference,
measured on the SAME tasks and the SAME seed, excludes zero.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_run_suite_trials, ()),
        ("exercise 2", _check_task_success_counts, ()),
        ("exercise 3", _check_pass_at_k, ()),
        ("exercise 4", _check_pass_hat_k, ()),
        ("exercise 5", _check_trajectory_metrics, ()),
        ("exercise 6", _check_failure_taxonomy_from_traces, ()),
        ("exercise 7", _check_paired_diff_interval, ()),
        ("exercise 8", _check_compare_configurations, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.